# 🧳 مساعد تخطيط السفر — AI Travel Planning Assistant
**مقرر أساسيات LangChain (LCH303) — دبلوم الذكاء الاصطناعي التوليدي — المجموعة 12**

يشغّل هذا الدفتر المشروع كاملاً على Google Colab:
1. تجهيز ملفات المشروع وتثبيت المكتبات
2. ربط مفتاح Groq بأمان من **Secrets** (دون كتابته في الكود)
3. تشغيل الاختبارات الآلية
4. استعراض قاعدة البيانات وقوالب التعليمات
5. عرض حي: إنشاء خطة ← تعديلها بالمحادثة ← حفظها وتصديرها
6. اختبار معالجة الأخطاء
7. تشغيل واجهة Streamlit برابط عام

> **قبل البدء:** من الشريط الجانبي في Colab افتح 🔑 **Secrets** وأضف مفتاحاً باسم `GROQ_API_KEY`
> (من https://console.groq.com/keys) وفعّل خيار الوصول للدفتر.

## 1) تجهيز ملفات المشروع
ارفع الملف المضغوط `travel_planner.zip` عند الطلب (أو ضعه في Google Drive وعدّل المسار).

In [ ]:
import os, zipfile
if not os.path.exists("travel_planner/app.py"):
    from google.colab import files
    uploaded = files.upload()                      # اختر travel_planner.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")
%cd travel_planner
!ls -R src tests | head -40

## 2) تثبيت المكتبات

In [ ]:
!pip install -q -r requirements.txt

## 3) ربط مفتاح Groq من Secrets

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ تم تحميل المفتاح من Secrets (لن يُطبع).")
except Exception as e:
    print("⚠️ لم يُعثر على المفتاح؛ سيعمل النظام بالخطة الاحتياطية فقط.", type(e).__name__)

## 4) تشغيل الاختبارات الآلية
35 اختباراً: قاعدة البيانات، التحقق، الميزانية، الخطة الاحتياطية، الاستيراد، ومكونات LangChain بنموذج وهمي.

In [ ]:
!python -m pytest -v

## 5) تهيئة الخدمة واستعراض قاعدة البيانات

In [ ]:
import sys, json, sqlite3
import pandas as pd
sys.path.insert(0, "src")
from travel_planner.planner import TravelPlannerService

service = TravelPlannerService()
print("النموذج:", service.settings.model_name, "| متاح:", service.llm_available)

con = sqlite3.connect(service.settings.db_path)
for t in ["destinations", "activities", "trips", "plan_items", "chat_messages"]:
    print(f"{t:15s}", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "صف")
display(pd.read_sql("SELECT * FROM destinations", con))
display(pd.read_sql("SELECT a.id, d.name AS الوجهة, a.name AS النشاط, a.category, a.cost_per_person, a.best_time "
                    "FROM activities a JOIN destinations d ON d.id=a.destination_id LIMIT 12", con))

## 6) قالب التعليمات بعد تعبئته (Prompt Engineering)
نعرض الرسالتين كما تصلان إلى النموذج: تعليمات النظام + بيانات الرحلة وجدول الأنشطة المسترجع.

In [ ]:
from travel_planner.prompts import planning_prompt
from travel_planner.formatting import activities_table
from travel_planner.chains import plan_parser

acts = service.db.search_activities("جدة", ["heritage", "food"])
messages = planning_prompt().format_messages(
    destination="جدة", destination_description="عروس البحر الأحمر", days=2, travelers=2,
    budget="4,000", base_cost="1,300", activities_budget="2,700", interests="تاريخ وتراث، مطاعم ومقاهي",
    notes="لا يوجد", max_per_day=4, activities_table=activities_table(acts),
    feedback="", format_instructions=plan_parser().get_format_instructions())
for m in messages:
    print(f"===== {m.type} =====\n{m.content[:1800]}\n")

## 7) عرض حي: إنشاء خطة بالنموذج اللغوي
تمر الخطة بـ: استرجاع ← قالب ← Groq ← محلل Pydantic ← تحقق من البيانات ← حساب الميزانية ← حفظ.

In [ ]:
result = service.create_plan({
    "destination": "المدينة المنورة", "days": 3, "travelers": 2, "budget": 6000,
    "interests": ["تاريخ وتراث", "مطاعم ومقاهي"], "notes": "نفضّل بداية هادئة صباحاً",
})
print("المصدر:", result.source, "| المحاولات:", result.attempts, "| رقم الرحلة:", result.trip_id)
print(service.export_text(result))

### المخرجات المنظمة (Structured Output) كما أنتجها المحلل

In [ ]:
print(json.dumps(result.plan.model_dump(), ensure_ascii=False, indent=2)[:2500])

## 8) تعديل الخطة بالمحادثة (الذاكرة)
طلبان متتاليان؛ الثاني يعتمد على سياق الأول المحفوظ في جدول `chat_messages`.

In [ ]:
r1 = service.revise_plan(result.trip_id, "استبدل نشاط التسوق إن وجد بنشاط تراثي، واجعل اليوم الأول أخف")
print(service.export_text(r1)[:1500])
r2 = service.revise_plan(result.trip_id, "ممتاز، الآن اجعل الخطة أوفر قدر الإمكان دون إلغاء ما عدّلته سابقاً")
print("الإجمالي قبل:", result.budget.total, "| بعد التعديلين:", r2.budget.total)
for m in service.chat_history(result.trip_id):
    print(f"[{m['role']}] {m['content'][:200]}")

## 9) حالات الاختبار ومعالجة الأخطاء

In [ ]:
from travel_planner.exceptions import TravelPlannerError
from dataclasses import replace

def attempt(title, fn):
    try:
        out = fn()
        print(f"✅ {title}: نجح ←", getattr(out, "source", out))
    except TravelPlannerError as e:
        print(f"🛑 {title}:\n   {e.user_message}")

base = {"destination": "جدة", "days": 2, "travelers": 2, "budget": 3000, "interests": ["تسوق"]}
attempt("أيام خارج النطاق", lambda: service.create_plan({**base, "days": 12}))
attempt("ميزانية سالبة", lambda: service.create_plan({**base, "budget": -10}))
attempt("وجهة غير متوفرة", lambda: service.create_plan({**base, "destination": "باريس"}))
attempt("اهتمام غير مدعوم", lambda: service.create_plan({**base, "interests": ["تزلج"]}))
attempt("ملف غير مدعوم", lambda: service.import_plan("plan.pdf", b"%PDF"))
attempt("JSON تالف", lambda: service.import_plan("plan.json", b"{bad"))
attempt("طلب تعديل فارغ", lambda: service.revise_plan(result.trip_id, ""))

# محاكاة عدم توفر API: خدمة بلا مفتاح ← خطة احتياطية مع تنبيه
offline = TravelPlannerService(settings=replace(service.settings, groq_api_key=None))
r = offline.create_plan({**base, "budget": 800})
print("\nبدون API ← المصدر:", r.source)
for w in r.warnings: print("   ⚠️", w)

## 10) الخطط المحفوظة في قاعدة البيانات

In [ ]:
display(pd.DataFrame(service.list_trips()))

## 11) تشغيل واجهة Streamlit برابط عام
يُنشأ رابط `trycloudflare.com` مؤقت؛ افتحه في المتصفح. أوقف الخلية لإيقاف الواجهة.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("🌐 رابط الواجهة:", m.group(0)); break